# Patient Data — Quality Check & Transformation
**Dataset:** Middle Eastern Patient Data 2021–2023  
**Purpose:** Validate data integrity before Power BI ingestion, then apply required transformations.

In [ ]:
import pandas as pd

df = pd.read_excel("fake_data_middle_eastern_patient_data_2021_2023_with_master_data.xlsx")

# Quick look at what we're working with
print(f"Shape: {df.shape}")
df.head(3)

In [ ]:
# Data types and nulls in one go
df.info()

In [ ]:
# Null check — all zeros, nothing to handle
df.isnull().sum()

In [ ]:
# Duplicate check — full row and UID level
print(f"Duplicate rows : {df.duplicated().sum()}")
print(f"Duplicate UIDs : {df.duplicated('UID').sum()}")

In [ ]:
# Validate categorical columns against expected values from dashboard
print(f"Nationalities  : {df['Nationality'].nunique()} unique — {sorted(df['Nationality'].unique())}")
print(f"Sex values     : {df['Sex'].unique().tolist()}")
print(f"Hospitals      : {df['Hospital Name'].nunique()} unique")
print(f"Diagnoses      : {df['Diagnosis Name'].nunique()} unique")

In [ ]:
# Check date coverage — should span 2021 to 2023 only
print(f"Date range: {df['Patient Visit Date'].min()} → {df['Patient Visit Date'].max()}")
out_of_range = df[~df['Patient Visit Date'].dt.year.isin([2021, 2022, 2023])]
print(f"Records outside 2021–2023: {len(out_of_range)}")

In [ ]:
# Age distribution — checking for outliers or impossible values
print(df['Age'].describe())
print(f"\nAge < 0   : {(df['Age'] < 0).sum()}")
print(f"Age > 120 : {(df['Age'] > 120).sum()}")

In [ ]:
# Whitespace audit on text columns — silent killers in groupBy operations
text_cols = ['Name', 'Nationality', 'Sex', 'Hospital Name', 'Diagnosis Name', 'Doctor Name']
whitespace_issues = {col: df[col].str.strip().ne(df[col]).sum() for col in text_cols}
print(whitespace_issues)
# All zeros — no trimming needed

In [ ]:
# Age stored in dataset vs computed from Birthdate — flagging discrepancies
# 376 rows are off by exactly 1 year — birthday hasn't occurred yet at time of visit
# Stored Age reflects age at start of visit year, not exact visit date
# Consistent with dashboard avg (52.7) — no correction needed
df['_computed_age'] = (df['Patient Visit Date'] - df['Birthdate']).dt.days // 365
age_mismatch = df[df['Age'] != df['_computed_age']]
print(f"Age mismatches : {len(age_mismatch)}")
print(f"Difference distribution:\n{(age_mismatch['_computed_age'] - age_mismatch['Age']).value_counts()}")
df.drop(columns=['_computed_age'], inplace=True)

---
**Quality check complete.** No nulls, no duplicates, no out-of-range dates, no impossible ages, no whitespace issues. The 376 age discrepancies are a known quirk (birthday not yet passed at time of visit) and do not affect dashboard metrics. Dataset is clear for use.

---
## Transformation — Add `Age Category`
Dashboard segments patients into four buckets. This field doesn't exist in the source — deriving it from `Age`.

In [ ]:
# Age category buckets aligned to dashboard: Children / Youth / Middle Age / Senior Citizen
bins   = [0, 17, 35, 59, 120]
labels = ['Children', 'Youth', 'Middle Age', 'Senior Citizen']

df['Age Category'] = pd.cut(df['Age'], bins=bins, labels=labels, right=True)

# Verify distribution against dashboard (Children ~2%, Youth ~29%, Middle Age ~41%, Senior ~28%)
df['Age Category'].value_counts(normalize=True).mul(100).round(2).sort_values()

In [ ]:
# Spot check
df[['Name', 'Age', 'Age Category']].sample(10, random_state=42)

In [ ]:
# Export — keeping original file intact
OUTPUT = "patient_data_cleaned_with_age_category.xlsx"
df.to_excel(OUTPUT, index=False)
print(f"Saved → {OUTPUT}  |  {df.shape[0]:,} rows × {df.shape[1]} columns")